In [1]:
import numpy as np

### 합성곱/풀링 계층 구현하기

In [2]:
x = np.random.rand(10,1,28,28)

In [3]:
x.shape

(10, 1, 28, 28)

In [4]:
x[0].shape

(1, 28, 28)

In [5]:
x[9][0]   # 10번째 데이터, 첫 번째 채널

array([[8.64928236e-01, 9.80281812e-01, 6.52799767e-01, 3.51176938e-01,
        5.01856880e-01, 8.42019020e-01, 5.94866193e-01, 1.50722424e-01,
        1.44567619e-01, 7.92177918e-01, 8.67958588e-01, 9.66624432e-01,
        6.50569984e-01, 7.60354045e-01, 6.54894011e-01, 1.24892046e-01,
        5.02110915e-01, 2.20548581e-01, 6.64010992e-02, 4.90544743e-01,
        6.50846604e-01, 1.61344766e-02, 5.41671329e-01, 9.94216912e-01,
        6.49085897e-01, 5.69008038e-01, 8.17848420e-01, 7.99900188e-01],
       [9.01002886e-01, 8.09556590e-01, 5.39443484e-02, 9.53954502e-01,
        8.03897683e-01, 7.46699782e-01, 4.26093123e-01, 4.05055677e-01,
        3.18297105e-01, 6.48165850e-01, 9.66504266e-01, 5.31954331e-01,
        1.41544296e-01, 3.83733487e-01, 9.97567869e-01, 8.56454929e-01,
        1.01647821e-01, 4.16013659e-01, 7.24558713e-01, 9.22776635e-01,
        7.85274704e-01, 1.88134849e-01, 2.74609348e-02, 3.74529122e-01,
        1.07871015e-01, 6.14073919e-01, 5.00537683e-01, 5.43066

### im2col로 데이터 전개하기

In [6]:
# im2col은 입력 데이터(특징 맵)를 필터링(가중치 계산)하기 좋게 전개하는(펼치는) 함수이다. 
# 3차원 입력 데이터에 im2col을 적용하면 2차원 행렬로 바뀐다. 배치 안의 데이터 수까지 포함하여 4차원 데이터를 2차원으로 변환한다.

def im2col(input_data, filter_h, filter_w, stride=1, pad=0): # input_data: (데이터 수, 채널 수, 높이, 너비)의 4차원 배열로 이루어진 입력 데이터, filter_h: 필터의 높이, filter_w: 필터의 너비, stride: 스트라이드, pad: 패딩
    N, C, H, W = input_data.shape               # N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = (H + 2*pad - filter_h)//stride + 1  # 출력 데이터의 높이 OH, fiter_h는 필터의 높이
    out_w = (W + 2*pad - filter_w)//stride + 1  # 출력 데이터의 너비 OW, filter_w는 필터의 너비

    img = np.pad(input_data, [(0,0), (0,0), (pad, pad), (pad, pad)], 'constant') # 입력 데이터에 pad만큼 0으로 채움
    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w)) # 입력 데이터를 2차원 행렬로 전개할 배열 0으로 초기화

    # 입력 이미지 데이터를 슬라이딩 윈도우 방식으로 필터링하여 2차원 배열로 변환
    # 필터의 높이, 너비만큼 이동하면서 필터를 적용
    for y in range(filter_h):
        y_max = y + stride*out_h
        for x in range(filter_w):
            x_max = x + stride*out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0, 4, 5, 1, 2, 3).reshape(N*out_h*out_w, -1)
    return col # 2차원 행렬로 전개


In [7]:
x1 = np.random.rand(1, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0
print(col1)

[[0.85572698 0.58296397 0.95459401 0.78241383 0.47045934 0.42348091
  0.42678304 0.30679479 0.94368647 0.68301041 0.20302456 0.21709464
  0.4106818  0.63687628 0.58068393 0.15155386 0.82170358 0.07348952
  0.33704301 0.4382633  0.88914934 0.14583731 0.81927416 0.52075585
  0.92521048 0.99747426 0.3231107  0.10239019 0.13335667 0.6046861
  0.86486201 0.0126646  0.96215839 0.55653785 0.24631372 0.60455967
  0.45765033 0.72050775 0.94452075 0.9637558  0.09592566 0.7460132
  0.23422416 0.57637888 0.98727702 0.79444653 0.78296205 0.94521132
  0.8679219  0.15665995 0.86765767 0.86536569 0.79287998 0.77073459
  0.25019094 0.94974072 0.99926801 0.41917857 0.53332855 0.61305471
  0.27253855 0.18704846 0.72957553 0.62902085 0.13986765 0.37571908
  0.75166109 0.3920874  0.62123428 0.10034748 0.85487267 0.63827522
  0.74630195 0.35346747 0.33831675]
 [0.58296397 0.95459401 0.78241383 0.47045934 0.20719893 0.42678304
  0.30679479 0.94368647 0.68301041 0.22490436 0.21709464 0.4106818
  0.63687628 0.

### 합성곱 계층 구현  

In [8]:
class Convolution:
  def __init__(self,W,b,stride=1,pad=0):
    self.W = W            # 필터(가중치)
    self.b = b            # 편향
    self.stride = stride  # 스트라이드
    self.pad = pad        # 패딩
  
  def forward(self,x):
    FN, C, FH, FW = self.W.shape  # 필터, FN: 필터 개수, C: 채널 수, FH: 필터 높이, FW: 필터 너비
    N, C, H, W = x.shape          # 입력 데이터, N: 데이터 수, C: 채널 수, H: 높이, W: 너비
    out_h = 1 + int((H + 2*self.pad - FH) / self.stride) # 출력 데이터 높이, OH
    out_w = 1 + int((W + 2*self.pad - FW) / self.stride) # 출력 데이터 너비, OW

    col = im2col(x, FH, FW, self.stride, self.pad) # 입력 데이터를 2차원 배열로 전개한 col
    col_W = self.W.reshape(FN, -1).T               # 필터를 reshape를 사용하여 2차원 배열로 전개한 col_W, T는 transpose를 의미

    out = np.dot(col, col_W) + self.b              # 합성곱 연산 수행 (행렬의 내적)
    out = out.reshape(N, out_h, out_w, -1).transpose(0, 3, 1, 2)  # 출력 데이터를 reshape하여 transpose, 형상을 (N(0), H(1), W(2), C(3))에서 (N(0), C(3), H(1), W(2))로 변경

    return out

In [ ]:
x1 = np.random.rand(1, 3, 7, 7) # (데이터 수, 채널 수, 높이, 너비), input_data
col1 = im2col(x1, 5, 5, stride=1, pad=0) # 필터 크기 5x5, 스트라이드 1, 패딩 0
print(col1)                              # input_data를 2차원 배열로 전개

# Convolution 클래스 사용
W = np.random.rand(3, 3, 5, 5) # (필터 수, 채널 수, 높이, 너비), filter 5x5x3
b = np.random.rand(1)          # 편향
conv1 = Convolution(W, b, stride=1, pad=0) # Convolution 클래스 생성

out1 = conv1.forward(x1) # 순전파
print(out1)              # iput_data에 필터를 적용한 결과 출력

### Pooling 계층  

In [ ]:
class Pooling:
  def __init__(self,pool_h,pool_w,stride=1,pad=0):
    self.pool_h = pool_h  # 풀링 높이
    self.pool_w = pool_w  # 풀링 너비
    self.stride = stride  # 스트라이드
    self.pad = pad        # 패딩
    
  def forward(self,x):
    N, C, H, W = x.shape
    out_h = int(1 + (H - self.pool_h) / self.stride) # 출력 데이터 높이 OH
    out_w = int(1 + (W - self.pool_w) / self.stride) # 출력 데이터 너비 OW
    
    # 전개(1)
    col = im2col(x, self.pool_h, self.pool_w, self.stride, self.pad)
    col = col.reshape(-1, self.pool_h*self.pool_w)
    
    
    